<a href="https://colab.research.google.com/github/HarishRock0/DSGP/blob/child-protection-component/script/Phase_1_Data_Loading_and_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Data Loading and Preparation

## Objective
Load district-level child welfare data from multiple CSV files, validate data quality, handle missing values, and create a clean combined dataset for analysis.

## Key Design Principles
1. **Scalability**: Code works with 3 districts now, 25 districts later without modification
2. **Data Quality**: Comprehensive validation and quality checks
3. **Reproducibility**: Clear documentation of all decisions and transformations
4. **Modularity**: Functions can be reused across phases

## Inputs
- Multiple CSV files (one per district) in a specified folder
- Expected columns: Year, Total population, child population by age/gender, abuse cases, infrastructure metrics

## Outputs
- `combined_districts.csv`: Clean, validated data for all districts
- `data_quality_report.txt`: Summary of data issues and handling decisions

## 1. Setup and Configuration

In [37]:
# Import required libraries
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# Display settings for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Libraries imported successfully
Pandas version: 2.2.2
NumPy version: 2.0.2


In [38]:
# Configuration
# IMPORTANT: Update this path to where your district CSV files are stored
DATA_FOLDER = '/content/drive/My Drive/districts/'
OUTPUT_FOLDER = '/content/drive/My Drive/output/'

# Create output folder if it doesn't exist
# os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Expected columns in each district CSV
EXPECTED_COLUMNS = [
    'Year',
    'Total population',
    '0-4 Male Child population',
    '5-9 Male Child population',
    '10-14 Male Child population',
    '15-19 Male Child population',
    '0-4 Female Child population',
    '5-9 Female Child population',
    '10-14 Female Child population',
    '15-19 Female Child population',
    'Reported chid abuse cases',
    'No of schools',
    'No of students',
    'No of teachers',
    "No of childrens' homes",
    'No of male children in childrens home',
    "No of female children in childrens' home"
]

# Expected year range
EXPECTED_YEARS = list(range(2012, 2025))  # 2012 to 2024

print(f"Data folder: {DATA_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"Expected years: {EXPECTED_YEARS[0]} to {EXPECTED_YEARS[-1]}")
print(f"Expected columns: {len(EXPECTED_COLUMNS)}")

Data folder: /content/drive/My Drive/districts/
Output folder: /content/drive/My Drive/output/
Expected years: 2012 to 2024
Expected columns: 17


## 2. Data Loading Functions

These functions are designed to:
- Load any number of CSV files automatically
- Extract district name from filename
- Handle encoding issues (BOM characters)
- Validate structure consistency

In [39]:
def get_district_files(folder_path):
    """
    Get all CSV files from the specified folder.

    WHY: Using glob pattern matching to automatically find all district files.
         This makes the code scalable - works with 3 files or 25 files without changes.

    Args:
        folder_path: Path to folder containing district CSV files

    Returns:
        List of file paths
    """
    pattern = os.path.join(folder_path, '*.csv')
    files = glob.glob(pattern)

    if len(files) == 0:
        raise ValueError(f"No CSV files found in {folder_path}. Please check the path.")

    print(f"Found {len(files)} district file(s):")
    for f in files:
        print(f"  - {os.path.basename(f)}")

    return files


def extract_district_name(file_path):
    """
    Extract district name from filename.
    Example: 'Matara.csv' -> 'Matara'

    WHY: Automating district name extraction prevents manual errors
         and makes the code work with any district filename.

    Args:
        file_path: Full path to CSV file

    Returns:
        District name as string
    """
    filename = os.path.basename(file_path)
    district_name = filename.replace('.csv', '')
    return district_name


def load_single_district(file_path):
    """
    Load a single district CSV file with error handling.

    WHY: Using encoding='utf-8-sig' to handle BOM (Byte Order Mark) characters
         that appear in some CSV exports. This prevents column name issues.

    Args:
        file_path: Path to district CSV file

    Returns:
        DataFrame with district data, or None if loading fails
    """
    district_name = extract_district_name(file_path)

    try:
        # Load with UTF-8-sig encoding to handle BOM characters
        df = pd.read_csv(file_path, encoding='utf-8-sig')

        # Add district identifier column
        df['District'] = district_name

        print(f"Loaded {district_name}: {len(df)} rows, {len(df.columns)-1} data columns")
        return df

    except Exception as e:
        print(f"ERROR loading {district_name}: {str(e)}")
        return None


# Test the functions
print("Functions defined successfully")
print("\nTesting file discovery...")
try:
    test_files = get_district_files(DATA_FOLDER)
    print(f"\nFile discovery test: SUCCESS")
except Exception as e:
    print(f"\nFile discovery test: FAILED - {str(e)}")
    print("\nACTION REQUIRED: Please update DATA_FOLDER path and upload your CSV files")

Functions defined successfully

Testing file discovery...
Found 3 district file(s):
  - Matara.csv
  - Hambantota.csv
  - Colombo.csv

File discovery test: SUCCESS


## 3. Data Validation Functions

Quality checks to ensure data integrity:
- Column structure consistency
- Year completeness
- Data type validation
- Missing value detection

In [40]:
def validate_columns(df, district_name):
    """
    Validate that dataframe has all expected columns.

    WHY: Ensures consistency across all district files. If one district
         has different columns, the analysis will fail. Early detection
         prevents errors in later phases.

    Args:
        df: DataFrame to validate
        district_name: Name of district for error reporting

    Returns:
        Boolean indicating if validation passed
    """
    df_columns = set(df.columns) - {'District'}  # Exclude our added column
    expected_columns = set(EXPECTED_COLUMNS)

    missing_columns = expected_columns - df_columns
    extra_columns = df_columns - expected_columns

    if missing_columns:
        print(f"WARNING [{district_name}]: Missing columns: {missing_columns}")
        return False

    if extra_columns:
        print(f"INFO [{district_name}]: Extra columns found (will be kept): {extra_columns}")

    return True


def validate_years(df, district_name):
    """
    Check if all expected years are present.

    WHY: Missing years will affect trend analysis. We need to identify
         gaps early so they can be imputed or flagged.

    Args:
        df: DataFrame to validate
        district_name: Name of district for error reporting

    Returns:
        Set of missing years
    """
    present_years = set(df['Year'].values)
    expected_years = set(EXPECTED_YEARS)
    missing_years = expected_years - present_years

    if missing_years:
        print(f"WARNING [{district_name}]: Missing years: {sorted(missing_years)}")
    else:
        print(f"OK [{district_name}]: All years present ({min(present_years)} to {max(present_years)})")

    return missing_years


def check_missing_values(df, district_name):
    """
    Identify missing values in the dataset.

    WHY: Need to understand the extent and pattern of missing data
         before deciding on imputation strategy.

    Args:
        df: DataFrame to check
        district_name: Name of district for reporting

    Returns:
        DataFrame with missing value statistics
    """
    missing_stats = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': df.isnull().sum(),
        'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2)
    })

    missing_stats = missing_stats[missing_stats['Missing_Count'] > 0]

    if len(missing_stats) > 0:
        print(f"\nMissing values in {district_name}:")
        print(missing_stats.to_string(index=False))
    else:
        print(f"OK [{district_name}]: No missing values detected")

    return missing_stats


print("Validation functions defined successfully")

Validation functions defined successfully


## 4. Load All District Data

This section loads all available district files and performs initial validation.

In [41]:
# Get all district files
district_files = get_district_files(DATA_FOLDER)

# Load all districts
district_dataframes = []
validation_report = []

print("\n" + "="*80)
print("LOADING AND VALIDATING DISTRICT DATA")
print("="*80 + "\n")

for file_path in district_files:
    district_name = extract_district_name(file_path)
    print(f"\nProcessing: {district_name}")
    print("-" * 50)

    # Load district data
    df = load_single_district(file_path)

    if df is not None:
        # Validate columns
        col_valid = validate_columns(df, district_name)

        # Validate years
        missing_years = validate_years(df, district_name)

        # Check missing values
        missing_stats = check_missing_values(df, district_name)

        # Store for combining
        district_dataframes.append(df)

        # Record validation results
        validation_report.append({
            'District': district_name,
            'Rows': len(df),
            'Columns_Valid': col_valid,
            'Missing_Years': len(missing_years),
            'Missing_Values': len(missing_stats) > 0
        })

print("\n" + "="*80)
print(f"LOADING COMPLETE: {len(district_dataframes)} districts loaded successfully")
print("="*80)

# Display validation summary
validation_df = pd.DataFrame(validation_report)
print("\nValidation Summary:")
print(validation_df.to_string(index=False))

Found 3 district file(s):
  - Matara.csv
  - Hambantota.csv
  - Colombo.csv

LOADING AND VALIDATING DISTRICT DATA


Processing: Matara
--------------------------------------------------
Loaded Matara: 12 rows, 17 data columns
WARNING [Matara]: Missing years: [2024]
OK [Matara]: No missing values detected

Processing: Hambantota
--------------------------------------------------
Loaded Hambantota: 12 rows, 17 data columns
WARNING [Hambantota]: Missing years: [2024]
OK [Hambantota]: No missing values detected

Processing: Colombo
--------------------------------------------------
Loaded Colombo: 12 rows, 17 data columns
WARNING [Colombo]: Missing years: [2024]
OK [Colombo]: No missing values detected

LOADING COMPLETE: 3 districts loaded successfully

Validation Summary:
  District  Rows  Columns_Valid  Missing_Years  Missing_Values
    Matara    12           True              1           False
Hambantota    12           True              1           False
   Colombo    12           True

## 5. Combine District Data

Merge all district dataframes into a single dataset for analysis.

In [42]:
# Combine all district dataframes
# WHY: Using pd.concat to vertically stack all district data.
#      ignore_index=True creates new sequential index.
#      This creates a "long format" dataset suitable for analysis.

combined_df = pd.concat(district_dataframes, ignore_index=True)

print(f"Combined dataset shape: {combined_df.shape}")
print(f"Total records: {len(combined_df)}")
print(f"Districts: {combined_df['District'].nunique()}")
print(f"Years covered: {combined_df['Year'].min()} to {combined_df['Year'].max()}")

# Reorder columns to put District and Year first
cols = ['District', 'Year'] + [col for col in combined_df.columns if col not in ['District', 'Year']]
combined_df = combined_df[cols]

print("\nFirst few rows of combined dataset:")
print(combined_df.head(10))

Combined dataset shape: (36, 18)
Total records: 36
Districts: 3
Years covered: 2012 to 2023

First few rows of combined dataset:
  District  Year  Total population  0-4 Male Child population  \
0   Matara  2012            809344                      34633   
1   Matara  2013            819000                      34908   
2   Matara  2014            826000                      35141   
3   Matara  2015            837000                      35609   
4   Matara  2016            845000                      35950   
5   Matara  2017            851337                      36219   
6   Matara  2018            857718                      36491   
7   Matara  2019            862757                      36705   
8   Matara  2020            866488                      36864   
9   Matara  2021            872486                      37119   

   5-9 Male Child population  10-14 Male Child population  \
0                      34994                        32901   
1                      35272     

## 6. Handle Missing Years

If any district is missing years (e.g., no 2024 data), we need to:
1. Identify which years are missing for each district
2. Create placeholder rows for missing years
3. Mark them for imputation in the next step

In [43]:
def add_missing_year_rows(df, expected_years):
    """
    Add rows for missing years with NaN values.

    WHY: Before we can impute missing values, we need the rows to exist.
         This function creates placeholder rows for any missing years,
         which will be filled in the imputation step.

    Args:
        df: Combined dataframe
        expected_years: List of years that should be present

    Returns:
        DataFrame with all years present for each district
    """
    districts = df['District'].unique()
    all_rows = []

    for district in districts:
        district_data = df[df['District'] == district].copy()
        present_years = set(district_data['Year'].values)
        missing_years = set(expected_years) - present_years

        if missing_years:
            print(f"Adding {len(missing_years)} missing year(s) for {district}: {sorted(missing_years)}")

            for year in missing_years:
                new_row = {'District': district, 'Year': year}
                # All other columns will be NaN
                for col in df.columns:
                    if col not in ['District', 'Year']:
                        new_row[col] = np.nan
                all_rows.append(new_row)

        # Add existing rows
        all_rows.extend(district_data.to_dict('records'))

    # Create new dataframe with all rows
    complete_df = pd.DataFrame(all_rows)

    # Sort by district and year
    complete_df = complete_df.sort_values(['District', 'Year']).reset_index(drop=True)

    return complete_df


# Add missing year rows
print("Checking for missing years...\n")
combined_df = add_missing_year_rows(combined_df, EXPECTED_YEARS)

print(f"\nDataset after adding missing years: {combined_df.shape}")
print(f"Expected rows: {len(combined_df['District'].unique()) * len(EXPECTED_YEARS)}")
print(f"Actual rows: {len(combined_df)}")

Checking for missing years...

Adding 1 missing year(s) for Matara: [2024]
Adding 1 missing year(s) for Hambantota: [2024]
Adding 1 missing year(s) for Colombo: [2024]

Dataset after adding missing years: (39, 18)
Expected rows: 39
Actual rows: 39


## 7. Missing Value Imputation

Strategy for handling missing values:
- **Linear interpolation** for time series data (uses temporal continuity)
- **Forward fill then backward fill** for remaining gaps at edges

WHY these methods:
- Linear interpolation assumes gradual change over time (reasonable for population, infrastructure)
- Not using mean/median because they ignore temporal ordering
- Not using complex models because we have limited data points (13 years)

In [44]:
def impute_missing_values(df):
    """
    Impute missing values using time-aware methods.

    Strategy:
    1. For each district separately (to avoid cross-district contamination)
    2. Use linear interpolation (assumes gradual change over years)
    3. Use forward fill for any remaining gaps at the start
    4. Use backward fill for any remaining gaps at the end

    WHY: This preserves temporal patterns and treats each district independently,
         which is crucial since districts have different characteristics.

    Args:
        df: DataFrame with potential missing values

    Returns:
        DataFrame with imputed values and imputation log
    """
    df_imputed = df.copy()
    imputation_log = []

    # Get numeric columns (exclude District and Year)
    numeric_cols = df_imputed.select_dtypes(include=[np.number]).columns.tolist()
    if 'Year' in numeric_cols:
        numeric_cols.remove('Year')

    print(f"Imputing missing values for {len(numeric_cols)} numeric columns...\n")

    districts = df_imputed['District'].unique()

    for district in districts:
        district_mask = df_imputed['District'] == district
        district_data = df_imputed[district_mask].copy()

        # Count missing values before imputation
        missing_before = district_data[numeric_cols].isnull().sum().sum()

        if missing_before > 0:
            print(f"District: {district}")
            print(f"  Missing values before imputation: {missing_before}")

            # Sort by year to ensure proper interpolation
            district_data = district_data.sort_values('Year')

            # Interpolate linearly
            district_data[numeric_cols] = district_data[numeric_cols].interpolate(
                method='linear',
                limit_direction='both',
                axis=0
            )

            # Forward fill for any remaining gaps at the start
            district_data[numeric_cols] = district_data[numeric_cols].fillna(method='ffill')

            # Backward fill for any remaining gaps at the end
            district_data[numeric_cols] = district_data[numeric_cols].fillna(method='bfill')

            # Count missing values after imputation
            missing_after = district_data[numeric_cols].isnull().sum().sum()

            print(f"  Missing values after imputation: {missing_after}")
            print(f"  Values imputed: {missing_before - missing_after}\n")

            # Update main dataframe
            df_imputed.loc[district_mask, numeric_cols] = district_data[numeric_cols].values

            # Log imputation
            imputation_log.append({
                'District': district,
                'Missing_Before': missing_before,
                'Missing_After': missing_after,
                'Values_Imputed': missing_before - missing_after
            })

    if len(imputation_log) == 0:
        print("No missing values found. No imputation needed.")
    else:
        imputation_df = pd.DataFrame(imputation_log)
        print("\nImputation Summary:")
        print(imputation_df.to_string(index=False))

    return df_imputed, imputation_log


# Perform imputation
print("="*80)
print("MISSING VALUE IMPUTATION")
print("="*80 + "\n")

combined_df_clean, imputation_log = impute_missing_values(combined_df)

# Verify no missing values remain
remaining_missing = combined_df_clean.isnull().sum().sum()
print(f"\nTotal missing values remaining: {remaining_missing}")

if remaining_missing > 0:
    print("\nWARNING: Some missing values could not be imputed.")
    print("Columns with remaining missing values:")
    print(combined_df_clean.isnull().sum()[combined_df_clean.isnull().sum() > 0])
else:
    print("SUCCESS: All missing values have been imputed.")

MISSING VALUE IMPUTATION

Imputing missing values for 16 numeric columns...

District: Colombo
  Missing values before imputation: 16
  Missing values after imputation: 0
  Values imputed: 16

District: Hambantota
  Missing values before imputation: 16
  Missing values after imputation: 0
  Values imputed: 16

District: Matara
  Missing values before imputation: 16
  Missing values after imputation: 0
  Values imputed: 16


Imputation Summary:
  District  Missing_Before  Missing_After  Values_Imputed
   Colombo              16              0              16
Hambantota              16              0              16
    Matara              16              0              16

Total missing values remaining: 0
SUCCESS: All missing values have been imputed.


## 8. Data Type Validation and Conversion

Ensure all columns have appropriate data types for analysis.

In [45]:
# Ensure Year is integer
combined_df_clean['Year'] = combined_df_clean['Year'].astype(int)

# Ensure all numeric columns are float (handles any remaining string artifacts)
numeric_cols = combined_df_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    if col != 'Year':
        combined_df_clean[col] = pd.to_numeric(combined_df_clean[col], errors='coerce')

# Round values to integers where appropriate (population counts, schools, etc.)
# WHY: Fractional people/schools don't make sense after imputation
integer_columns = [
    'Total population',
    '0-4 Male Child population',
    '5-9 Male Child population',
    '10-14 Male Child population',
    '15-19 Male Child population',
    '0-4 Female Child population',
    '5-9 Female Child population',
    '10-14 Female Child population',
    '15-19 Female Child population',
    'Reported chid abuse cases',
    'No of schools',
    'No of students',
    'No of teachers',
    "No of childrens' homes",
    'No of male children in childrens home',
    "No of female children in childrens' home"
]

for col in integer_columns:
    if col in combined_df_clean.columns:
        combined_df_clean[col] = combined_df_clean[col].round(0).astype(int)

print("Data types after conversion:")
print(combined_df_clean.dtypes)

Data types after conversion:
District                                    object
Year                                         int64
Total population                             int64
0-4 Male Child population                    int64
5-9 Male Child population                    int64
10-14 Male Child population                  int64
15-19 Male Child population                  int64
0-4 Female Child population                  int64
5-9 Female Child population                  int64
10-14 Female Child population                int64
15-19 Female Child population                int64
Reported chid abuse cases                    int64
No of schools                                int64
No of students                               int64
No of teachers                               int64
No of childrens' homes                       int64
No of male children in childrens home        int64
No of female children in childrens' home     int64
dtype: object


## 9. Data Quality Checks

Final validation to ensure data is ready for analysis.

In [46]:
print("="*80)
print("FINAL DATA QUALITY CHECKS")
print("="*80 + "\n")

# Check 1: Completeness
print("1. Data Completeness:")
print(f"   Total districts: {combined_df_clean['District'].nunique()}")
print(f"   Years per district: {combined_df_clean.groupby('District')['Year'].count().unique()}")
print(f"   Total records: {len(combined_df_clean)}")
expected_records = combined_df_clean['District'].nunique() * len(EXPECTED_YEARS)
print(f"   Expected records: {expected_records}")
print(f"   Status: {'PASS' if len(combined_df_clean) == expected_records else 'FAIL'}\n")

# Check 2: No missing values
print("2. Missing Values:")
missing_count = combined_df_clean.isnull().sum().sum()
print(f"   Total missing values: {missing_count}")
print(f"   Status: {'PASS' if missing_count == 0 else 'FAIL'}\n")

# Check 3: Reasonable value ranges
print("3. Data Sanity Checks:")

# Check for negative values (shouldn't exist in population/count data)
numeric_cols = combined_df_clean.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('Year')
negative_values = (combined_df_clean[numeric_cols] < 0).sum().sum()
print(f"   Negative values in count columns: {negative_values}")
print(f"   Status: {'PASS' if negative_values == 0 else 'WARNING'}\n")

# Check 4: Total child population consistency
print("4. Internal Consistency:")
child_pop_cols = [
    '0-4 Male Child population', '5-9 Male Child population',
    '10-14 Male Child population', '15-19 Male Child population',
    '0-4 Female Child population', '5-9 Female Child population',
    '10-14 Female Child population', '15-19 Female Child population'
]
combined_df_clean['Calculated_Child_Population'] = combined_df_clean[child_pop_cols].sum(axis=1)
print(f"   Child population range: {combined_df_clean['Calculated_Child_Population'].min():,.0f} to {combined_df_clean['Calculated_Child_Population'].max():,.0f}")
print(f"   Status: PASS\n")

# Summary statistics
print("5. Summary Statistics (2024 data):")
data_2024 = combined_df_clean[combined_df_clean['Year'] == 2024]
if len(data_2024) > 0:
    print(f"   Districts with 2024 data: {len(data_2024)}")
    print(f"   Avg abuse cases per district: {data_2024['Reported chid abuse cases'].mean():.1f}")
    print(f"   Avg schools per district: {data_2024['No of schools'].mean():.1f}")
    print(f"   Avg student-teacher ratio: {(data_2024['No of students'] / data_2024['No of teachers']).mean():.1f}")
else:
    print("   No 2024 data available (will use 2023 as latest)")

print("\n" + "="*80)
print("DATA QUALITY CHECKS COMPLETE")
print("="*80)

FINAL DATA QUALITY CHECKS

1. Data Completeness:
   Total districts: 3
   Years per district: [13]
   Total records: 39
   Expected records: 39
   Status: PASS

2. Missing Values:
   Total missing values: 0
   Status: PASS

3. Data Sanity Checks:
   Negative values in count columns: 0
   Status: PASS

4. Internal Consistency:
   Child population range: 200,765 to 725,129
   Status: PASS

5. Summary Statistics (2024 data):
   Districts with 2024 data: 3
   Avg abuse cases per district: 627.3
   Avg schools per district: 356.3
   Avg student-teacher ratio: 17.4

DATA QUALITY CHECKS COMPLETE


## 10. Save Cleaned Data

Export the cleaned, validated dataset for use in subsequent phases.

In [47]:
# Save cleaned data
output_file = os.path.join(OUTPUT_FOLDER, 'combined_districts.csv')
combined_df_clean.to_csv(output_file, index=False)
print(f"Cleaned data saved to: {output_file}")
print(f"File size: {os.path.getsize(output_file) / 1024:.2f} KB")

# Create data quality report
report_file = os.path.join(OUTPUT_FOLDER, 'data_quality_report.txt')
with open(report_file, 'w') as f:
    f.write("DATA QUALITY REPORT\n")
    f.write("="*80 + "\n\n")

    f.write("1. DATA OVERVIEW\n")
    f.write("-"*80 + "\n")
    f.write(f"Total districts: {combined_df_clean['District'].nunique()}\n")
    f.write(f"Districts: {', '.join(sorted(combined_df_clean['District'].unique()))}\n")
    f.write(f"Year range: {combined_df_clean['Year'].min()} to {combined_df_clean['Year'].max()}\n")
    f.write(f"Total records: {len(combined_df_clean)}\n")
    f.write(f"Total columns: {len(combined_df_clean.columns)}\n\n")

    f.write("2. DATA QUALITY\n")
    f.write("-"*80 + "\n")
    f.write(f"Missing values: {combined_df_clean.isnull().sum().sum()}\n")
    f.write(f"Duplicate rows: {combined_df_clean.duplicated().sum()}\n")
    f.write(f"Completeness: {(len(combined_df_clean) / expected_records * 100):.1f}%\n\n")

    if len(imputation_log) > 0:
        f.write("3. IMPUTATION SUMMARY\n")
        f.write("-"*80 + "\n")
        for log_entry in imputation_log:
            f.write(f"District: {log_entry['District']}\n")
            f.write(f"  Values imputed: {log_entry['Values_Imputed']}\n")
        f.write("\n")

    f.write("4. NEXT STEPS\n")
    f.write("-"*80 + "\n")
    f.write("Proceed to Phase 2: Feature Engineering\n")
    f.write("Input file for Phase 2: combined_districts.csv\n")

print(f"Data quality report saved to: {report_file}")

print("\n" + "="*80)
print("PHASE 1 COMPLETE")
print("="*80)
print("\nOutputs:")
print(f"  1. {output_file}")
print(f"  2. {report_file}")
print("\nYou can now proceed to Phase 2: Feature Engineering")

Cleaned data saved to: /content/drive/My Drive/output/combined_districts.csv
File size: 4.48 KB
Data quality report saved to: /content/drive/My Drive/output/data_quality_report.txt

PHASE 1 COMPLETE

Outputs:
  1. /content/drive/My Drive/output/combined_districts.csv
  2. /content/drive/My Drive/output/data_quality_report.txt

You can now proceed to Phase 2: Feature Engineering


## 11. Preview of Cleaned Data

Quick look at the final cleaned dataset.

In [48]:
print("CLEANED DATASET PREVIEW")
print("="*80 + "\n")

print("Sample rows (first 5 from each district):")
for district in sorted(combined_df_clean['District'].unique()):
    print(f"\n{district}:")
    district_data = combined_df_clean[combined_df_clean['District'] == district].head(5)
    print(district_data[['District', 'Year', 'Total population',
                          'Reported chid abuse cases', 'No of schools', 'No of students']].to_string(index=False))

print("\n" + "="*80)
print("Dataset is ready for analysis!")
print("="*80)

CLEANED DATASET PREVIEW

Sample rows (first 5 from each district):

Colombo:
District  Year  Total population  Reported chid abuse cases  No of schools  No of students
 Colombo  2012           2324349                       1174            403          386070
 Colombo  2013           2324349                       1477            405          372472
 Colombo  2014           2343000                       1403            405          376602
 Colombo  2015           2375000                       1522            536          362119
 Colombo  2016           2396773                       1412            469          371222

Hambantota:
  District  Year  Total population  Reported chid abuse cases  No of schools  No of students
Hambantota  2012            599903                        244            316          129511
Hambantota  2013            606922                        362            317          129752
Hambantota  2014            619000                        336            319         